# FitSync Module 4 — CatVTON Worker Tesla T4 Smoke Test (Isolated Python 3.10 + CUDA 12.1 Compiler + PyTorch 2.4.0)

This reproducible notebook validates the **actual `runpod-worker/` implementation** on a fresh Google Colab **NVIDIA Tesla T4 GPU** runtime — even when Colab defaults to **CUDA Toolkit 13.0 (`nvcc V13.0.88`)** and `cuda-toolkit-12-1` is unavailable from APT.

### Architecture & CUDA Compatibility Fix
1. **Isolated Python 3.10 Venv (`/content/catvton-env`):** Pinned `torch==2.4.0+cu121`, `torchvision==0.19.0+cu121`, `torchaudio==2.4.0+cu121`.
2. **Isolated Matching CUDA 12.1 Compiler (`/content/cuda-12.1`):** Installed strictly from NVIDIA's official `nvidia/label/cuda-12.1.1` conda channel via `micromamba` (`cuda-compiler=12.1.1`, `cuda-cudart-dev=12.1.105`, `cuda-cccl=12.1.109`, `libcublas-dev`, `libcusparse-dev`, `libcusolver-dev`, `libcurand-dev`) — completely independent of system APT CUDA repos or Colab's `/usr/local/cuda` (13.0).
3. **Compatible Host Compiler (`GCC <= 12`):** Automatically checks host `g++` version (and configures `gcc-12`/`g++-12` when Colab runs GCC 13+ on Ubuntu 24.04) so `nvcc 12.1.105` compiles Detectron2 and SCHP `inplace_abn` cleanly.
4. **Cross-Cell Persistence (`_catvton_cuda_env.pth`):** Persists `CUDA_HOME=/content/cuda-12.1`, `PATH`, `CC`/`CXX`, `TORCH_EXTENSIONS_DIR`, and `TORCH_CUDA_ARCH_LIST` via a `.pth` startup hook inside `/content/catvton-env` (immune to Ubuntu `/usr/lib/python3.10/sitecustomize.py` shadowing) so both **Cell 1** and **Cell 4** always match CUDA `12.1`.
5. **No Repo Shadowing:** Clones and builds Detectron2 in `/tmp/detectron2-src` and strips `Zheng-Chong/CatVTON`'s prebuilt Python 3.9 `detectron2/` (`_C.cpython-39*.so`) and `densepose/` folders so Python 3.10 always loads the freshly compiled `catvton-env` extensions.

> **Before Running Cell 2:** Upload `runpod-worker/colab_bundle.zip` to `/content/colab_bundle.zip` in the Colab Files sidebar if `runpod-worker/` has not yet been pushed to GitHub.

In [ ]:
%%bash
set -euo pipefail

echo "=== [1/7] Checking NVIDIA GPU Runtime ==="
if ! command -v nvidia-smi &> /dev/null; then
    echo "ERROR: No NVIDIA GPU runtime detected. In Colab go to Runtime -> Change runtime type -> T4 GPU." >&2
    exit 1
fi
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
SYS_NVCC=$(nvcc --version 2>/dev/null | grep -oE 'release [0-9]+\.[0-9]+' || echo 'not installed')
echo "System nvcc before isolation: ${SYS_NVCC}"

echo "=== [2/7] Checking Base Build Tools, Python 3.10, and Host GCC (<= 12 for CUDA 12.1) ==="
NEED_APT=0
for cmd in curl bzip2 ninja git gcc g++; do
    if ! command -v "$cmd" &> /dev/null; then
        NEED_APT=1
    fi
done
if ! command -v python3.10 &> /dev/null || ! python3.10 -m venv --help &> /dev/null; then
    NEED_APT=1
fi
GXX_MAJOR=$((g++ -dumpfullversion 2>/dev/null || g++ -dumpversion 2>/dev/null || echo 0) | head -n1 | cut -d. -f1)
if [ "$GXX_MAJOR" -gt 12 ] && ! command -v g++-12 &> /dev/null; then
    NEED_APT=1
fi

if [ "$NEED_APT" -eq 1 ]; then
    export DEBIAN_FRONTEND=noninteractive
    apt-get update -qq
    apt-get install -y -qq software-properties-common build-essential ninja-build git curl bzip2 ca-certificates
    if ! command -v python3.10 &> /dev/null || ! python3.10 -m venv --help &> /dev/null; then
        add-apt-repository -y ppa:deadsnakes/ppa || true
        apt-get update -qq
        apt-get install -y -qq python3.10 python3.10-venv python3.10-dev
    fi
    GXX_MAJOR=$((g++ -dumpfullversion 2>/dev/null || g++ -dumpversion) | head -n1 | cut -d. -f1)
    if [ "$GXX_MAJOR" -gt 12 ] && ! command -v g++-12 &> /dev/null; then
        apt-get install -y -qq gcc-12 g++-12
    fi
fi

echo "=== [3/7] Installing Isolated CUDA 12.1 Compiler (/content/cuda-12.1 via nvidia/label/cuda-12.1.1) ==="
if [ ! -x "/content/cuda-12.1/bin/nvcc" ]; then
    mkdir -p /content/bin
    if [ ! -x "/content/bin/micromamba" ]; then
        curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    fi
    if [ ! -x "/content/bin/micromamba" ]; then
        echo "ERROR: Failed to extract micromamba into /content/bin/micromamba." >&2
        exit 1
    fi
    export MAMBA_ROOT_PREFIX="/content/micromamba"
    /content/bin/micromamba create -y -q -p /content/cuda-12.1 \
        --override-channels -c nvidia/label/cuda-12.1.1 \
        cuda-compiler=12.1.1 \
        cuda-cudart-dev=12.1.105 \
        cuda-cccl=12.1.109 \
        cuda-driver-dev=12.1.105 \
        cuda-nvtx=12.1.105 \
        libcublas-dev=12.1.3.1 \
        libcusparse-dev=12.1.0.106 \
        libcusolver-dev=11.4.5.107 \
        libcurand-dev=10.3.2.106
fi
ln -sfn /content/cuda-12.1/lib /content/cuda-12.1/lib64

mkdir -p /content/cuda-12.1/host-bin
GXX_MAJOR=$((g++ -dumpfullversion 2>/dev/null || g++ -dumpversion) | head -n1 | cut -d. -f1)
if [ "$GXX_MAJOR" -gt 12 ]; then
    if ! command -v g++-12 &> /dev/null || ! command -v gcc-12 &> /dev/null; then
        echo "ERROR: Host g++ is GCC ${GXX_MAJOR} (> 12), which CUDA 12.1 nvcc rejects, and gcc-12/g++-12 could not be installed." >&2
        exit 1
    fi
    ln -sfn "$(command -v gcc-12)" /content/cuda-12.1/host-bin/gcc
    ln -sfn "$(command -v g++-12)" /content/cuda-12.1/host-bin/g++
else
    ln -sfn "$(command -v gcc)" /content/cuda-12.1/host-bin/gcc
    ln -sfn "$(command -v g++)" /content/cuda-12.1/host-bin/g++
fi

export CUDA_HOME="/content/cuda-12.1"
export CUDA_PATH="/content/cuda-12.1"
export PATH="/content/cuda-12.1/host-bin:/content/cuda-12.1/bin:$PATH"
export CC="/content/cuda-12.1/host-bin/gcc"
export CXX="/content/cuda-12.1/host-bin/g++"
export CUDAHOSTCXX="/content/cuda-12.1/host-bin/g++"
export FORCE_CUDA="1"
export TORCH_EXTENSIONS_DIR="/content/catvton-env/.torch_extensions"

echo "Isolated nvcc: $(/content/cuda-12.1/bin/nvcc --version | tr '\n' ' ')"
echo "Host C++ compiler for nvcc: $($CXX --version | head -n 1)"

echo "=== [4/7] Creating Isolated Python 3.10 Environment & Persisting CUDA 12.1 Config ==="
if [ ! -d "/content/catvton-env" ]; then
    python3.10 -m venv /content/catvton-env
fi

VENV_PIP="/content/catvton-env/bin/pip"
VENV_PY="/content/catvton-env/bin/python"

$VENV_PIP install -q --upgrade "pip<25.0" "setuptools<70.0.0" wheel
$VENV_PY --version

SITE_PKGS=$($VENV_PY -c "import site; print(site.getsitepackages()[-1])")
cat << 'EOF' > "${SITE_PKGS}/_catvton_cuda_env.py"
import os

_cuda_home = "/content/cuda-12.1"
_host_bin = "/content/cuda-12.1/host-bin"
_cuda_bin = "/content/cuda-12.1/bin"
if os.path.isdir(_cuda_home):
    os.environ["CUDA_HOME"] = _cuda_home
    os.environ["CUDA_PATH"] = _cuda_home
    _path_parts = [p for p in os.environ.get("PATH", "").split(os.pathsep) if p and p not in (_host_bin, _cuda_bin)]
    os.environ["PATH"] = os.pathsep.join([_host_bin, _cuda_bin] + _path_parts)
    if os.path.exists(os.path.join(_host_bin, "g++")):
        os.environ["CC"] = os.path.join(_host_bin, "gcc")
        os.environ["CXX"] = os.path.join(_host_bin, "g++")
        os.environ["CUDAHOSTCXX"] = os.path.join(_host_bin, "g++")
    os.environ.setdefault("FORCE_CUDA", "1")
    os.environ.setdefault("TORCH_EXTENSIONS_DIR", "/content/catvton-env/.torch_extensions")
    os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "7.5+PTX")
EOF
echo "import _catvton_cuda_env" > "${SITE_PKGS}/_catvton_cuda_env.pth"
echo "import _catvton_cuda_env  # noqa: F401" > "${SITE_PKGS}/sitecustomize.py"

echo "=== [5/7] Installing Pinned PyTorch 2.4.0+cu121 & Worker Dependencies ==="
$VENV_PIP install -q \
    torch==2.4.0+cu121 \
    torchvision==0.19.0+cu121 \
    torchaudio==2.4.0+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

$VENV_PIP install -q \
    numpy==1.26.4 \
    Pillow==10.4.0 \
    opencv-python-headless==4.10.0.84 \
    scipy==1.13.1 \
    scikit-image==0.24.0 \
    matplotlib==3.9.0 \
    tqdm==4.66.4 \
    PyYAML==6.0.1 \
    packaging==24.1 \
    pydantic==2.8.2 \
    requests==2.32.3 \
    diffusers==0.29.2 \
    transformers==4.42.4 \
    accelerate==0.31.0 \
    huggingface-hub==0.23.4 \
    safetensors==0.4.3 \
    einops==0.8.0 \
    omegaconf==2.3.0 \
    fvcore==0.1.5.post20221221 \
    iopath==0.1.9 \
    pycocotools==2.0.8 \
    av==12.3.0 \
    cloudpickle==3.0.0 \
    yacs==0.1.8 \
    tabulate==0.9.0 \
    termcolor==2.4.0 \
    ninja==1.11.1.1 \
    runpod==1.7.7

echo "=== [6/7] Verifying PyTorch <-> CUDA Compiler Alignment & .pth Override ==="
ARCH_LIST=$(CUDA_HOME=/usr/local/cuda CUDA_PATH=/usr/local/cuda PATH="/usr/local/cuda/bin:$PATH" $VENV_PY - << 'PYEOF'
import os, re, subprocess, sys, torch

if not torch.cuda.is_available():
    sys.exit("ERROR: torch.cuda.is_available() is False inside /content/catvton-env.")

cuda_home = os.environ.get("CUDA_HOME", "")
if cuda_home != "/content/cuda-12.1":
    sys.exit(f"ERROR: Expected _catvton_cuda_env.pth to force CUDA_HOME='/content/cuda-12.1', got {cuda_home!r}.")
nvcc = os.path.join(cuda_home, "bin", "nvcc")
if not os.path.isfile(nvcc):
    sys.exit(f"ERROR: nvcc not found at {nvcc!r}.")

out = subprocess.check_output([nvcc, "--version"], text=True)
m = re.search(r"release (\d+\.\d+)", out)
nvcc_ver = m.group(1) if m else "unknown"
torch_cuda = torch.version.cuda or "unknown"

if nvcc_ver != torch_cuda:
    sys.exit(
        f"ERROR: CUDA compiler mismatch! nvcc={nvcc_ver} ({nvcc}) vs torch.version.cuda={torch_cuda}."
    )

major, minor = torch.cuda.get_device_capability(0)
arch = f"{major}.{minor}+PTX"
print(
    f"Verified alignment: GPU={torch.cuda.get_device_name(0)} (sm_{major}{minor}) | "
    f"torch={torch.__version__} | torch_cuda={torch_cuda} | nvcc={nvcc_ver} | CUDA_HOME={cuda_home}",
    file=sys.stderr,
)
print(arch)
PYEOF
)
export TORCH_CUDA_ARCH_LIST="$ARCH_LIST"
echo "Using TORCH_CUDA_ARCH_LIST=${TORCH_CUDA_ARCH_LIST}"

echo "=== [7/7] Compiling Detectron2 + DensePose, Cloning CatVTON, and Pre-Compiling SCHP inplace_abn ==="
rm -rf /content/detectron2
if ! (cd /tmp && $VENV_PY -c "import os, detectron2, detectron2._C, densepose; assert os.path.abspath(detectron2._C.__file__).startswith('/content/catvton-env'); assert os.path.abspath(densepose.__file__).startswith('/content/catvton-env')" &> /dev/null); then
    rm -rf /tmp/detectron2-src
    $VENV_PIP uninstall -y detectron2 detectron2-densepose &> /dev/null || true
    git clone --depth 1 https://github.com/facebookresearch/detectron2.git /tmp/detectron2-src
    (cd /tmp && $VENV_PIP install --no-cache-dir --no-build-isolation /tmp/detectron2-src)
    (cd /tmp && $VENV_PIP install --no-cache-dir --no-build-isolation --no-deps /tmp/detectron2-src/projects/DensePose)
    rm -rf /tmp/detectron2-src
fi

(cd /tmp && $VENV_PY -c "import os, detectron2, detectron2._C, densepose; print('Detectron2 _C OK:', detectron2._C.__file__); print('DensePose OK:', densepose.__file__); assert os.path.abspath(detectron2._C.__file__).startswith('/content/catvton-env'), detectron2._C.__file__")

if [ ! -d "/content/CatVTON" ]; then
    git clone --depth 1 https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON
fi
# Zheng-Chong/CatVTON bundles a prebuilt Python 3.9 detectron2/_C.cpython-39*.so and densepose/ at repo root;
# remove them so PYTHONPATH=/content/CatVTON never shadows the Python 3.10 + CUDA 12.1 build in catvton-env.
rm -rf /content/CatVTON/detectron2 /content/CatVTON/densepose

mkdir -p /content/densepose_/tmp /content/CatVTON/densepose_/tmp
PYTHONPATH=/content/CatVTON $VENV_PY -c "
import os, sys, torch, diffusers, transformers, accelerate, detectron2, detectron2._C, densepose
assert os.path.abspath(detectron2._C.__file__).startswith('/content/catvton-env'), detectron2._C.__file__
from model.SCHP import SCHP
if os.path.isdir('/content/CatVTON/model/SCHP/modules'):
    from model.SCHP.modules import functions as schp_functions
    schp_mode = f'inplace_abn={schp_functions._backend.__file__}'
else:
    schp_mode = 'native BatchNorm2d (CatVTON edited branch)'
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
print('READY:', sys.version.split()[0], '| torch:', torch.__version__, '| torch.version.cuda:', torch.version.cuda, '| detectron2:', detectron2.__version__, '| SCHP:', schp_mode, '| GPU:', torch.cuda.get_device_name(0))
"

In [ ]:
# Cell 2: Load runpod-worker/ from uploaded colab_bundle.zip (or Git repository)
import os
import shutil
import subprocess
import zipfile

REPO_URL = "https://github.com/Darknight67456/fitsynnpc.git"
BRANCH = "feature/module4-ai-tryon"
TARGET_WORKER_DIR = "/content/CatVTON/runpod-worker"
BUNDLE_CANDIDATES = ["/content/colab_bundle.zip", "/content/CatVTON/colab_bundle.zip"]

os.makedirs(TARGET_WORKER_DIR, exist_ok=True)

bundle_found = next((p for p in BUNDLE_CANDIDATES if os.path.isfile(p)), None)
if bundle_found:
    print(f"Extracting uploaded worker bundle: {bundle_found} -> {TARGET_WORKER_DIR}")
    with zipfile.ZipFile(bundle_found, "r") as zf:
        zf.extractall(TARGET_WORKER_DIR)
else:
    print(f"No colab_bundle.zip found locally; cloning {REPO_URL} (branch: {BRANCH})...")
    clone_dir = "/content/fitsynnpc_repo"
    if os.path.exists(clone_dir):
        shutil.rmtree(clone_dir)
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, clone_dir], check=True)
    repo_worker_dir = os.path.join(clone_dir, "runpod-worker")
    if os.path.isdir(repo_worker_dir):
        shutil.copytree(repo_worker_dir, TARGET_WORKER_DIR, dirs_exist_ok=True)
        print(f"Synced runpod-worker/ from branch '{BRANCH}'.")
    else:
        raise FileNotFoundError(
            f"'runpod-worker/' is not pushed to {REPO_URL} ({BRANCH}) yet and no colab_bundle.zip was uploaded. "
            "Upload 'runpod-worker/colab_bundle.zip' to '/content/colab_bundle.zip' in the Colab Files sidebar and re-run this cell."
        )

required_files = [
    "preprocessor.py",
    "fitsync_mask.py",
    "catvton_pipeline.py",
    "handler.py",
    "test_worker.py",
    "gpu_smoke_test.py",
    "samples/mannequin_front_768x1024.png",
    "samples/jersey_padini_768x1024.png",
]
for rel in required_files:
    full_p = os.path.join(TARGET_WORKER_DIR, rel)
    if not os.path.exists(full_p):
        raise FileNotFoundError(f"Missing required worker file: {full_p}. Re-upload updated colab_bundle.zip.")
    print(f"  [OK] {rel} ({os.path.getsize(full_p):,} bytes)")

In [ ]:
# Cell 3: Run Worker Dependency Verification & Unit Tests in Isolated Python 3.10 Environment
import os
import subprocess

PYTHON_BIN = "/content/catvton-env/bin/python"
CUDA121_HOME = "/content/cuda-12.1"
if not os.path.isfile(PYTHON_BIN):
    raise FileNotFoundError(f"{PYTHON_BIN} not found. Run Cell 1 first to build the isolated Python 3.10 environment.")

dep_check = (
    "import os, torch, detectron2, detectron2._C, densepose, diffusers, transformers, accelerate, huggingface_hub; "
    "assert os.path.abspath(detectron2._C.__file__).startswith('/content/catvton-env'), detectron2._C.__file__; "
    "print('Deps OK:', torch.__version__, 'CUDA', torch.version.cuda, '| CUDA_HOME:', os.environ.get('CUDA_HOME'), '| detectron2', detectron2.__version__, '| _C:', detectron2._C.__file__)"
)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["MPLBACKEND"] = "Agg"  # isolated Python 3.10 cannot use matplotlib_inline backend
if os.path.isfile(os.path.join(CUDA121_HOME, "bin", "nvcc")):
    env["CUDA_HOME"] = CUDA121_HOME
    env["CUDA_PATH"] = CUDA121_HOME
    _prepend = [p for p in (os.path.join(CUDA121_HOME, "host-bin"), os.path.join(CUDA121_HOME, "bin")) if os.path.isdir(p)]
    env["PATH"] = os.pathsep.join(_prepend + [p for p in env.get("PATH", "").split(os.pathsep) if p and p not in _prepend])

for cmd in ([PYTHON_BIN, "-c", dep_check], [PYTHON_BIN, "/content/CatVTON/runpod-worker/test_worker.py"]):
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

In [ ]:
# Cell 4: Run Real AutoMasker (DensePose + SCHP) + CatVTON GPU Inference on Tesla T4
import os
import subprocess

PYTHON_BIN = "/content/catvton-env/bin/python"
CUDA121_HOME = "/content/cuda-12.1"
CUDA121_NVCC = os.path.join(CUDA121_HOME, "bin", "nvcc")
samples_dir = "/content/CatVTON/runpod-worker/samples"
person_path = os.path.join(samples_dir, "mannequin_front_768x1024.png")
garment_path = os.path.join(samples_dir, "jersey_padini_768x1024.png")
mask_out = os.path.join(samples_dir, "gpu_smoke_mask.png")
image_out = os.path.join(samples_dir, "gpu_smoke_output.png")
person_out = os.path.join(samples_dir, "gpu_smoke_person.png")  # reframed mannequin actually sent to CatVTON
SLEEVE_LENGTH = "short_sleeve"  # Jersey Padini is short-sleeved -> protect forearms

for required in (PYTHON_BIN, CUDA121_NVCC, person_path, garment_path):
    if not os.path.isfile(required):
        raise FileNotFoundError(f"Missing prerequisite: {required}. Ensure Cell 1 (Python 3.10 + CUDA 12.1) and Cell 2 completed cleanly.")

env = os.environ.copy()
env["CATVTON_REPO_PATH"] = "/content/CatVTON"
env["PYTHONUNBUFFERED"] = "1"
env["MPLBACKEND"] = "Agg"  # isolated Python 3.10 cannot use matplotlib_inline backend
env["CUDA_HOME"] = CUDA121_HOME
env["CUDA_PATH"] = CUDA121_HOME
host_bin = os.path.join(CUDA121_HOME, "host-bin")
cuda_bin = os.path.join(CUDA121_HOME, "bin")
prepend = [p for p in (host_bin, cuda_bin) if os.path.isdir(p)]
env["PATH"] = os.pathsep.join(prepend + [p for p in env.get("PATH", "").split(os.pathsep) if p and p not in prepend])
if os.path.isfile(os.path.join(host_bin, "gcc")):
    env["CC"] = os.path.join(host_bin, "gcc")
if os.path.isfile(os.path.join(host_bin, "g++")):
    env["CXX"] = os.path.join(host_bin, "g++")
    env["CUDAHOSTCXX"] = os.path.join(host_bin, "g++")
env.setdefault("FORCE_CUDA", "1")
env.setdefault("TORCH_EXTENSIONS_DIR", "/content/catvton-env/.torch_extensions")
env.setdefault("TORCH_CUDA_ARCH_LIST", "7.5+PTX")

cmd = [
    PYTHON_BIN,
    "/content/CatVTON/runpod-worker/gpu_smoke_test.py",
    "--require-gpu",
    "--person", person_path,
    "--garment", garment_path,
    "--category", "upper_body",
    "--sleeve-length", SLEEVE_LENGTH,
    "--mask-refinement", "fitsync",
    "--steps", "50",
    "--guidance", "2.5",
    "--seed", "42",
    "--output-mask", mask_out,
    "--output-image", image_out,
    "--output-person", person_out,
]
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)

for out_path in (mask_out, image_out, person_out):
    if not os.path.isfile(out_path):
        raise FileNotFoundError(f"Cell 4 exited 0 but did not write expected file: {out_path}")
print(f"Verified Cell 4 outputs:\n  - {person_out}\n  - {mask_out}\n  - {image_out}")

In [ ]:
# Cell 5: Inspect & Save 4-Panel Verification Artifact (Reframed Input Mannequin, Jersey Padini, AutoMasker Mask, CatVTON Output)
import os
from PIL import Image
from IPython.display import display

samples_dir = "/content/CatVTON/runpod-worker/samples"
required_files = {
    "person": os.path.join(samples_dir, "gpu_smoke_person.png"),  # reframed input actually used
    "garment": os.path.join(samples_dir, "jersey_padini_768x1024.png"),
    "mask": os.path.join(samples_dir, "gpu_smoke_mask.png"),
    "output": os.path.join(samples_dir, "gpu_smoke_output.png"),
}
missing = [f"{k}: {p}" for k, p in required_files.items() if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(
        "Missing prerequisite image(s) for Cell 5 comparison grid:\n  "
        + "\n  ".join(missing)
        + "\nRun Cells 1-4 in order and verify Cell 4 exited with code 0 before running Cell 5."
    )

person_img = Image.open(required_files["person"]).convert("RGB")
garment_img = Image.open(required_files["garment"]).convert("RGB")
mask_img = Image.open(required_files["mask"]).convert("RGB")
output_img = Image.open(required_files["output"]).convert("RGB")

grid = Image.new("RGB", (768 * 4, 1024))
for idx, img in enumerate([person_img, garment_img, mask_img, output_img]):
    grid.paste(img, (idx * 768, 0))

comparison_path = os.path.join(samples_dir, "gpu_smoke_comparison.png")
grid.save(comparison_path)
print(f"Saved 4-panel comparison to: {comparison_path}")
display(grid.resize((768 * 2, 512)))